# 🌳 Geo-AI Training — Workshop 2: CHM และการนับต้นไม้

**แนวคิด**: คำนวณ `CHM = DSM − DTM` เพื่อได้ความสูงของสิ่งที่อยู่เหนือพื้นดิน แล้วหาจุดยอด (peak) เพื่อระบุตำแหน่งต้นไม้แต่ละต้น

## 🧩 Setup


> 📦 ก่อนรัน ติดตั้ง dependencies ให้ครบ
> pip install -r requirements.txt
> ```

In [ ]:
# ติดตั้ง library ที่ใช้ในไฟล์นี้ (ใช้เวลาสักครู่ตอนรันครั้งแรก)
!pip install -q rasterio scikit-image scikit-learn geopandas shapely folium gdown

print("ติดตั้งเสร็จแล้ว ✅")

### ติดตั้งฟอนต์ไทย (สำหรับกราฟที่มีข้อความไทย)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import urllib.request
import os

# โหลดฟอนต์ Sarabun (ฟอนต์ไทยจาก Google Fonts) มาใช้กับกราฟ — โหลดครั้งเดียว ถ้ามีไฟล์แล้วข้ามได้เลย
font_path = "Sarabun-Regular.ttf"
if not os.path.exists(font_path):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/google/fonts/main/ofl/sarabun/Sarabun-Regular.ttf",
        font_path
    )
fm.fontManager.addfont(font_path)
plt.rcParams["font.family"] = "Sarabun"
plt.rcParams["axes.unicode_minus"] = False

print("ติดตั้งฟอนต์ไทยเสร็จแล้ว ✅")

In [ ]:
import os

# 📁 ใช้พื้นที่เก็บไฟล์ชั่วคราวในเครื่อง Colab (ไม่เชื่อม Google Drive)
# ⚠️ ไฟล์ในนี้จะหายไปเมื่อ Colab runtime ถูกตัดการเชื่อมต่อ/รีสตาร์ท — ดาวน์โหลดเก็บเองก่อนปิดเครื่อง
# (ใช้แผง Files ด้านซ้ายของ Colab คลิกขวาไฟล์ > Download)
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/outputs"
EXPORT_DIR = os.path.join(OUTPUT_DIR, "export")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print("DATA_DIR   :", DATA_DIR)
print("EXPORT_DIR :", EXPORT_DIR)

### โหลดข้อมูลจาก Google Drive

ดาวน์โหลดข้อมูลโดรนจริง 3 ไฟล์ (ถ้ามีอยู่แล้วใน `DATA_DIR` จะข้ามการโหลดซ้ำ)

In [ ]:
import gdown

FILES = {
    "orthophoto.tif": "1i3yprs-Es03CFbMKx2rTvcsXJs0KOX_r",
    "dtm.tif": "1FMfmeasnKm7XkqdH7OrBk-3ZCcXaNHzZ",
    "dsm.tif": "1tL881oAJjaDWaYKgADQYD9fjtm_incm0",
}

for filename, file_id in FILES.items():
    out_path = os.path.join(DATA_DIR, filename)
    if os.path.exists(out_path):
        print(f"✅ มีไฟล์อยู่แล้ว: {filename}")
    else:
        print(f"⬇️  กำลังโหลด: {filename} ...")
        gdown.download(id=file_id, output=out_path, quiet=False)

print("\nเสร็จแล้ว พร้อมใช้งาน")

## 🔬 คำนวณ CHM และตรวจจับต้นไม้

In [ ]:
import rasterio
import numpy as np

with rasterio.open(os.path.join(DATA_DIR, "dsm.tif")) as src:
    dsm = src.read(1).astype(np.float32)
    dsm_nodata = src.nodata
    dsm_profile = src.profile
with rasterio.open(os.path.join(DATA_DIR, "dtm.tif")) as src:
    dtm = src.read(1).astype(np.float32)
    dtm_nodata = src.nodata

# กรองพิกเซล NoData ออกก่อน ไม่งั้น CHM จะเพี้ยนมหาศาล
if dsm_nodata is not None:
    dsm[dsm == dsm_nodata] = np.nan
if dtm_nodata is not None:
    dtm[dtm == dtm_nodata] = np.nan

chm = dsm - dtm
chm = np.nan_to_num(chm, nan=0.0)  # จุดที่ไม่มีข้อมูล ถือว่าไม่มีต้นไม้ (0 เมตร)
chm[chm < 0] = 0  # กันค่าติดลบจาก noise เล็กน้อย

H, W = chm.shape
print(f"ขนาด CHM: {H} x {W} พิกเซล")

### วิธีเดิม (baseline): หาต้นไม้จากความสูงอย่างเดียว

หาจุดยอด (local maxima) ในภาพ CHM ที่สูงเกิน `MIN_HEIGHT` — ยังไม่แยกว่าจุดนั้นเป็นต้นไม้หรือสิ่งปลูกสร้าง

In [ ]:
from skimage.feature import peak_local_max

MIN_HEIGHT = 10.0     # 🔧 ปรับได้: ความสูงขั้นต่ำที่นับเป็น "ต้นไม้" (เมตร)
MIN_DISTANCE = 10     # 🔧 ปรับได้: ระยะห่างขั้นต่ำระหว่างจุดยอด (พิกเซล)

coords_baseline = peak_local_max(chm, min_distance=MIN_DISTANCE, threshold_abs=MIN_HEIGHT)
print(f"วิธีเดิม (ใช้แค่ความสูงอย่างเดียว) พบจุดยอดทั้งหมด {len(coords_baseline)} จุด")
print("⚠️ ในจำนวนนี้อาจมีตึก/สิ่งปลูกสร้างปนอยู่ เพราะยังไม่ได้แยกพืชออกจากสิ่งปลูกสร้าง")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
im = axes[0].imshow(chm, cmap="YlGn")
axes[0].set_title("CHM (ความสูงเหนือพื้นดิน)")
plt.colorbar(im, ax=axes[0], fraction=0.046, label="เมตร")

axes[1].imshow(chm, cmap="YlGn")
axes[1].scatter(coords_baseline[:, 1], coords_baseline[:, 0], c="red", s=15, marker="x")
axes[1].set_title(f"วิธีเดิม (ความสูงอย่างเดียว): {len(coords_baseline)} จุด")
for a in axes:
    a.axis("off")
plt.tight_layout()
plt.show()

> ⚠️ **ปัญหา**: สิ่งปลูกสร้างก็มีความสูงเหนือพื้นดินเหมือนกัน วิธีข้างบนจึงอาจตรวจจับตึก/หลังคาเป็น "ต้นไม้" ปลอมได้
>
> **วิธีแก้**: ใช้ orthophoto (ภาพสี) มาช่วยแยกว่าจุดที่สูงนั้นเป็น "พืช" จริงหรือไม่ ก่อนนับเป็นต้นไม้ — แต่มี 2 เรื่องที่ต้องระวังก่อน:
> 1. **orthophoto กับ DSM/DTM คนละระบบพิกัดกัน** — orthophoto เป็น EPSG:4326 (lat/lon องศา) ส่วน DSM/DTM เป็น UTM (เมตร) และคนละกริดด้วย ต้อง **reproject** ให้ตรงกันก่อนจะเอามาใช้ร่วมกัน
> 2. **ข้อมูลนี้เป็นกล้อง RGB ธรรมดา ไม่มีแชนแนล Near-Infrared (NIR)** จึงคำนวณ NDVI แบบมาตรฐานไม่ได้ ต้องใช้ vegetation index ที่คำนวณจาก RGB ล้วนแทน เช่น **ExG (Excess Green Index)**

### โหลด Orthophoto แล้ว reproject ให้ตรงกริดกับ CHM

In [ ]:
from rasterio.warp import reproject, Resampling

with rasterio.open(os.path.join(DATA_DIR, "orthophoto.tif")) as src:
    ortho_src = src.read()  # (4, orig_h, orig_w) = Red, Green, Blue, Alpha
    ortho_src_crs = src.crs
    ortho_src_transform = src.transform

# ปลายทาง: array ขนาดเดียวกับ CHM (H, W) ในระบบพิกัดของ DSM/DTM
ortho_on_dsm = np.zeros((4, H, W), dtype=np.uint8)
reproject(
    source=ortho_src,
    destination=ortho_on_dsm,
    src_transform=ortho_src_transform,
    src_crs=ortho_src_crs,
    dst_transform=dsm_profile["transform"],
    dst_crs=dsm_profile["crs"],
    resampling=Resampling.bilinear,
)

print("Reproject orthophoto ให้ตรงกริดกับ CHM เสร็จแล้ว ✅")
print("ขนาด orthophoto ที่ reproject แล้ว:", ortho_on_dsm.shape)

### คำนวณ Vegetation Index จาก RGB (ExG)

`ExG = 2G - R - B` — พืชสีเขียวจะมีค่าสูงกว่าตึก/ถนน/ดินอย่างชัดเจน แม้ไม่มีแชนแนล NIR ก็ใช้แยกพืชได้ในระดับหนึ่ง

In [ ]:
valid = ortho_on_dsm[3] > 0  # แชนแนล alpha บอกว่าพิกเซลไหนมีข้อมูลจริง (ไม่ใช่ขอบว่างจากการ reproject)

r = ortho_on_dsm[0].astype(np.float32)
g = ortho_on_dsm[1].astype(np.float32)
b = ortho_on_dsm[2].astype(np.float32)
exg = 2 * g - r - b

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(np.where(valid, exg, np.nan), cmap="RdYlGn")
plt.colorbar(im, ax=ax, fraction=0.046, label="ExG")
ax.set_title("Vegetation Index (ExG)")
ax.axis("off")
plt.tight_layout()
plt.show()

### หา threshold อัตโนมัติด้วย Otsu แล้วสร้าง vegetation mask

Otsu เป็นวิธีหาค่า threshold อัตโนมัติจากฮิสโตแกรมของภาพ (แบ่งพิกเซลเป็น 2 กลุ่มที่ต่างกันชัดที่สุด) ไม่ต้องเดาค่าเอง

In [ ]:
from skimage.filters import threshold_otsu

otsu_thresh = threshold_otsu(exg[valid])
veg_mask = (exg > otsu_thresh) & valid

print(f"threshold ที่หาได้อัตโนมัติ (Otsu): ExG > {otsu_thresh:.1f}")
print(f"พื้นที่ที่จัดเป็นพืช: {veg_mask.mean()*100:.1f}% ของภาพ")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(ortho_on_dsm[:3].transpose(1, 2, 0))
axes[0].set_title("Orthomosaic (reproject แล้ว)")
axes[1].imshow(veg_mask, cmap="Greens")
axes[1].set_title("Vegetation Mask (พื้นที่พืช)")
for a in axes:
    a.axis("off")
plt.tight_layout()
plt.show()

### กรอง CHM ด้วย vegetation mask แล้วหาต้นไม้ใหม่

จุดที่ไม่ใช่พืช (เช่น ตึก) จะถูกตัดออกจาก CHM ก่อนหาจุดยอด ไม่ว่าจะสูงแค่ไหนก็จะไม่ถูกนับเป็นต้นไม้

In [ ]:
chm_veg = np.where(veg_mask, chm, 0)

tree_coords = peak_local_max(chm_veg, min_distance=MIN_DISTANCE, threshold_abs=MIN_HEIGHT)
tree_ys, tree_xs = tree_coords[:, 0], tree_coords[:, 1]
tree_heights = chm[tree_ys, tree_xs]

print(f"วิธีเดิม (ความสูงอย่างเดียว): {len(coords_baseline)} จุด")
print(f"วิธีใหม่ (ความสูง + vegetation index): {len(tree_xs)} จุด")
print(f"จำนวนจุดที่ถูกกรองออก (คาดว่าเป็นสิ่งปลูกสร้าง): {len(coords_baseline) - len(tree_xs)} จุด")
if len(tree_xs) > 0:
    print(f"ความสูงต้นไม้เฉลี่ย: {tree_heights.mean():.2f} m, สูงสุด: {tree_heights.max():.2f} m")

In [ ]:
# เทียบจุดที่ "ถูกกรองออก" (มีอยู่ในวิธีเดิม แต่ไม่ผ่าน vegetation mask) กับจุดที่เหลือ (ต้นไม้จริง)
filtered_set = set(map(tuple, tree_coords))
removed_coords = np.array([c for c in coords_baseline if tuple(c) not in filtered_set])

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(ortho_on_dsm[:3].transpose(1, 2, 0))
if len(removed_coords) > 0:
    ax.scatter(removed_coords[:, 1], removed_coords[:, 0], c="red", s=20, marker="x",
               label=f"ถูกกรองออก ({len(removed_coords)} จุด)")
ax.scatter(tree_xs, tree_ys, c="cyan", s=15, marker="o",
           label=f"ต้นไม้ที่ยืนยัน ({len(tree_xs)} จุด)")
ax.legend(loc="upper right")
ax.set_title("เปรียบเทียบ: ก่อน (แดง=ตัดออก) vs หลัง (ฟ้า=ต้นไม้จริง)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 💾 Export ผลลัพธ์

In [ ]:
import geopandas as gpd

# บันทึก CHM เป็น GeoTIFF
chm_profile = dsm_profile.copy()
chm_profile.update(dtype="float32")
with rasterio.open(os.path.join(EXPORT_DIR, "chm.tif"), "w", **chm_profile) as dst:
    dst.write(chm.astype(np.float32), 1)

print("✅ Export แล้ว: chm.tif")

In [ ]:
# บันทึก vegetation mask เป็น GeoTIFF
mask_profile = dsm_profile.copy()
mask_profile.update(dtype="uint8", nodata=0)
with rasterio.open(os.path.join(EXPORT_DIR, "vegetation_mask.tif"), "w", **mask_profile) as dst:
    dst.write(veg_mask.astype(np.uint8), 1)

print("✅ Export แล้ว: vegetation_mask.tif")

In [ ]:
from shapely.geometry import Point

# บันทึกตำแหน่งต้นไม้ (หลังกรองด้วย vegetation mask แล้ว) เป็น GeoJSON
tree_points = []
for px, py, h in zip(tree_xs, tree_ys, tree_heights):
    wx, wy = dsm_profile["transform"] * (int(px), int(py))
    tree_points.append({"geometry": Point(wx, wy), "height_m": round(float(h), 2)})

trees_gdf = gpd.GeoDataFrame(tree_points, crs=dsm_profile["crs"])
trees_gdf.to_file(os.path.join(EXPORT_DIR, "trees.geojson"), driver="GeoJSON")

print("✅ Export แล้ว: trees.geojson")

## 🌍 แผนที่ Interactive (Folium)

ดูผลลัพธ์ทั้งหมดซ้อนกันบนแผนที่ เลื่อน/ซูม/คลิกดูรายละเอียด และเปิด-ปิดแต่ละ layer ได้: orthomosaic, CHM, vegetation mask, และตำแหน่งต้นไม้ที่ตรวจพบ

In [ ]:
import folium
from rasterio.transform import array_bounds
from rasterio.warp import transform_bounds as warp_transform_bounds

# แปลงขอบเขตของกริด DSM/DTM (UTM) เป็น lat/lon (EPSG:4326) ที่ folium ใช้วางแผนที่
dsm_west, dsm_south, dsm_east, dsm_north = array_bounds(H, W, dsm_profile["transform"])
lon_min, lat_min, lon_max, lat_max = warp_transform_bounds(
    dsm_profile["crs"], "EPSG:4326", dsm_west, dsm_south, dsm_east, dsm_north)
image_bounds = [[lat_min, lon_min], [lat_max, lon_max]]

m = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2],
                zoom_start=17, tiles="CartoDB positron")
print("สร้างแผนที่ฐานเรียบร้อย ✅")

In [ ]:
# 🔧 ปรับได้: ย่อภาพก่อนวางบนแผนที่ เพื่อไม่ให้หน้าเว็บหนักเกินไป (ภาพเต็มมีหลายสิบล้านพิกเซล)
PREVIEW_STEP = max(1, max(H, W) // 1000)

ortho_preview = ortho_on_dsm[:3, ::PREVIEW_STEP, ::PREVIEW_STEP].transpose(1, 2, 0)

folium.raster_layers.ImageOverlay(
    image=ortho_preview, bounds=image_bounds, name="Orthomosaic", opacity=1.0,
).add_to(m)

print("เพิ่ม layer Orthomosaic แล้ว ✅")

In [ ]:
# แปลง CHM เป็นภาพสี (colormap) ก่อนวางบนแผนที่ (folium ต้องการภาพ ไม่ใช่ array ตัวเลขดิบ)
chm_preview = chm[::PREVIEW_STEP, ::PREVIEW_STEP]
chm_norm = np.clip(chm_preview / max(chm_preview.max(), 1e-6), 0, 1)
chm_rgb_preview = (plt.get_cmap("YlGn")(chm_norm)[:, :, :3] * 255).astype(np.uint8)

folium.raster_layers.ImageOverlay(
    image=chm_rgb_preview, bounds=image_bounds, name="CHM", opacity=0.6, show=False,
).add_to(m)

print("เพิ่ม layer CHM แล้ว ✅")

In [ ]:
# vegetation mask เป็นภาพโปร่งแสง (RGBA) เขียว = พื้นที่พืช, โปร่งใส = ไม่ใช่พืช
veg_preview = veg_mask[::PREVIEW_STEP, ::PREVIEW_STEP]
veg_rgba_preview = np.zeros((*veg_preview.shape, 4), dtype=np.uint8)
veg_rgba_preview[veg_preview] = [40, 180, 40, 160]

folium.raster_layers.ImageOverlay(
    image=veg_rgba_preview, bounds=image_bounds, name="Vegetation Mask", opacity=0.8, show=False,
).add_to(m)

print("เพิ่ม layer Vegetation Mask แล้ว ✅")

In [ ]:
# ตำแหน่งต้นไม้ที่ตรวจพบ — แสดงเป็นจุดวงกลมทั้งหมดโดยตรง (ไม่จัดกลุ่มเป็นตัวเลข)
trees_4326 = trees_gdf.to_crs(4326)
tree_layer = folium.FeatureGroup(name="ต้นไม้ที่ตรวจพบ")

for _, row in trees_4326.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x], radius=3,
        color="green", weight=1, fill=True, fill_color="lightgreen", fill_opacity=0.8,
        popup=f"ความสูง: {row.height_m} m",
    ).add_to(tree_layer)

tree_layer.add_to(m)
print(f"เพิ่ม layer ต้นไม้ ({len(trees_4326)} ต้น) แล้ว ✅")

In [ ]:
folium.LayerControl(collapsed=False).add_to(m)
m

---
✅ **จบ Workshop 2** — ไปต่อที่ `3_workshop3_flood.ipynb`